In [1]:
from pathlib import Path
import re
import subprocess
import tempfile
from pytubefix import YouTube


def download_audio(link, path="audio_pytubefix"):
    video = YouTube(link)
    stream = video.streams.get_audio_only()
    if stream is None:
        raise ValueError("No audio stream found")

    output_dir = Path(path)
    output_dir.mkdir(parents=True, exist_ok=True)
    wav_file = output_dir / f"{video.video_id}.wav"
    if wav_file.exists():
        print(f"Already downloaded: {wav_file}")
        return

    with tempfile.TemporaryDirectory(dir=output_dir) as temp_dir:
        source_file = stream.download(output_path=temp_dir, filename="audio")
        if not source_file:
            raise RuntimeError("Download did not produce an audio file")
        temp_wav = Path(temp_dir) / "audio.wav"
        subprocess.run(
            ["ffmpeg", "-y", "-i", source_file, "-vn", "-acodec", "pcm_s16le", str(temp_wav)],
            check=True,
        )
        temp_wav.rename(wav_file)

    print(f"Saved: {wav_file}")


In [5]:
cookies_from_browser = "chrome"

In [6]:
for directory in (Path.cwd(), *Path.cwd().parents):
    podcasts_file = directory / "data" / "podcasts.md"
    if podcasts_file.is_file():
        break
else:
    raise FileNotFoundError("Could not find data/podcasts.md")

links = re.findall(r"https?://[^\s)>]+", podcasts_file.read_text(encoding="utf-8"))
print(f"Found {len(links)} links")


Found 30 links


In [7]:
failed_downloads = []

for index, link in enumerate(links, start=1):
    print(f"({index}/{len(links)}) {link}")
    try:
        download_audio(link)
    except Exception as error:
        failed_downloads.append(link)
        print(f"Failed: {error}")

print(f"Finished: {len(links) - len(failed_downloads)} succeeded, {len(failed_downloads)} failed.")


(1/30) https://youtu.be/Ww1RrOeTq78?si=NOjK9xj5WiITcQ5o
Failed: Ww1RrOeTq78 This request was detected as a bot. DO NOT OPEN AN ISSUE! See more details at https://pytubefix.readthedocs.io/en/latest/user/po_token.html
(2/30) https://youtu.be/5-iWpLLHSEE?si=une4FAbaGVsVth1b
Failed: 5-iWpLLHSEE This request was detected as a bot. DO NOT OPEN AN ISSUE! See more details at https://pytubefix.readthedocs.io/en/latest/user/po_token.html
(3/30) https://youtu.be/9N3GvCTJCKQ?si=a_fzn3qD02g-lUp_
Failed: 9N3GvCTJCKQ This request was detected as a bot. DO NOT OPEN AN ISSUE! See more details at https://pytubefix.readthedocs.io/en/latest/user/po_token.html
(4/30) https://youtu.be/XUnEvzI8VQY?si=Irhr5vRrwYKXdbBA
Failed: XUnEvzI8VQY This request was detected as a bot. DO NOT OPEN AN ISSUE! See more details at https://pytubefix.readthedocs.io/en/latest/user/po_token.html
(5/30) https://youtu.be/g97I8Fvdkmc?si=YSyMPFT6xF49aL3H
Failed: g97I8Fvdkmc This request was detected as a bot. DO NOT OPEN AN ISSUE! Se